# Driver Drowsiness Detection - Full Final Notebook
Cleaned and corrected notebook ready for GitHub upload.


In [ ]:
# STEP 1 - IMPORT LIBRARIES
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import shuffle
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

IMG_SIZE = 64
print('Libraries Imported Successfully')
print('TensorFlow Version:', tf.__version__)


In [ ]:
# STEP 2 - DEFINE DATASET PATHS
source_folders = {
    'eyes/train/Close': r'C:\\dataset\\eyes\\train\\Close',
    'eyes/train/Open': r'C:\\dataset\\eyes\\train\\Open',
    'yawn/yawn': r'C:\\dataset\\yawn\\yawn',
    'yawn/no_yawn': r'C:\\dataset\\yawn\\no_yawn'
}

label_map = {
    'eyes/train/Close': 1,
    'eyes/train/Open': 0,
    'yawn/yawn': 1,
    'yawn/no_yawn': 0
}

print('Paths Loaded Successfully')


In [ ]:
# STEP 3 - LOAD IMAGES
X = []
y = []
MAX_PER_FOLDER = 4000

for key, path in source_folders.items():
    label = label_map[key]
    if not os.path.isdir(path):
        continue

    count = 0
    for file in os.listdir(path):
        if count >= MAX_PER_FOLDER:
            break

        file_path = os.path.join(path, file)
        img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            continue

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img / 255.0

        X.append(img)
        y.append(label)
        count += 1

X = np.array(X, dtype=np.float32)
y = np.array(y)

X = X.reshape(X.shape[0], -1)

print('Dataset Shape:', X.shape)
print('Labels Shape:', y.shape)


In [ ]:
# STEP 4 - TRAIN TEST SPLIT
X, y = shuffle(X, y, random_state=42)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print('Train:', X_train.shape[0])
print('Validation:', X_val.shape[0])
print('Test:', X_test.shape[0])


In [ ]:
# STEP 5 - BUILD ANN MODEL
model = Sequential([
    Dense(1024, activation='relu', input_shape=(IMG_SIZE * IMG_SIZE,)),
    Dropout(0.4),
    Dense(512, activation='relu'),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
# STEP 6 - TRAIN MODEL
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
# STEP 7 - PLOT RESULTS
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Accuracy')
plt.legend(['Train','Validation'])

plt.subplot(1,2,2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Loss')
plt.legend(['Train','Validation'])

plt.show()


In [ ]:
# STEP 8 - TEST MODEL
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print('Test Accuracy:', acc * 100)


In [ ]:
# STEP 9 - REPORT
y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
# STEP 10 - SAVE MODEL
model.save('drowsiness_ann_model.keras')
print('Model Saved Successfully')
